# Visualizations for linear regression preparation

This notebook only creates exploratory visualizations. It does not train a model. The goal is to understand whether the merged data has useful patterns, missing values, outliers, and approximately linear relationships that could later support a linear regression approach.

## Data used

- `outputs/gemeente_odin_2024.csv`: municipality-level travel indicators from ODiN 2024. The main variable of interest is `bike_share_pct`.
- `outputs/buurt_master.csv`: neighbourhood-level CBS and PC6 data. These variables are aggregated to municipality level before visualization.
- Merge key: `gm_code`, for example `GM0014`. This is the shared municipality ID.

The visual question is: which municipality characteristics appear related to cycling share, and which variables look usable for a later regression?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 80)

BASE = Path.cwd()
OUTPUTS = BASE / 'outputs'
FIGURES = BASE / 'figures_linear_regression'
FIGURES.mkdir(exist_ok=True)

print(BASE)
print('Outputs folder exists:', OUTPUTS.exists())

## 1. Load merged outputs

In [ ]:
buurt_master = pd.read_csv(OUTPUTS / 'buurt_master.csv')
gemeente_odin = pd.read_csv(OUTPUTS / 'gemeente_odin_2024.csv')

print('buurt_master:', buurt_master.shape)
print('gemeente_odin:', gemeente_odin.shape)
display(buurt_master.head())
display(gemeente_odin.head())

## 2. Aggregate neighbourhood variables to municipality level

The ODiN table is already municipality-level. The CBS neighbourhood table is aggregated to the same municipality level using `gm_code`. Counts are summed. Continuous neighbourhood characteristics are population-weighted averages using `a_inw` as weight.

In [ ]:
def weighted_average(group, value_col, weight_col='a_inw'):
    values = pd.to_numeric(group[value_col], errors='coerce')
    weights = pd.to_numeric(group[weight_col], errors='coerce').fillna(0)
    mask = values.notna() & weights.gt(0)
    if not mask.any():
        return np.nan
    return np.average(values[mask], weights=weights[mask])

weighted_cols = ['bev_dich', 'g_wozbag', 'p_koopw', 'p_huurw', 'g_ink_pi', 'g_afs_hp', 'g_afs_gs', 'g_afs_kv', 'g_afs_sc', 'g_3km_sc', 'ste_mvs']
sum_cols = ['a_inw', 'a_woning', 'a_opp_ha', 'pc6_rows', 'pc6_unique_approx']

municipality_rows = []
for gm_code, group in buurt_master.groupby('gm_code'):
    row = {
        'gm_code': gm_code,
        'municipality_name': group['Gemeentenaam2025'].dropna().iloc[0] if group['Gemeentenaam2025'].notna().any() else np.nan,
        'n_neighbourhoods': len(group),
    }
    for col in sum_cols:
        row[col + '_sum'] = pd.to_numeric(group[col], errors='coerce').sum(min_count=1)
    for col in weighted_cols:
        row[col + '_wavg'] = weighted_average(group, col)
    municipality_rows.append(row)

municipality_features = pd.DataFrame(municipality_rows)
visual_df = gemeente_odin.merge(municipality_features, on='gm_code', how='inner')

visual_df['log_population'] = np.log1p(visual_df['a_inw_sum'])
visual_df['dwellings_per_1000_residents'] = visual_df['a_woning_sum'] / visual_df['a_inw_sum'] * 1000
visual_df['pc6_rows_per_1000_residents'] = visual_df['pc6_rows_sum'] / visual_df['a_inw_sum'] * 1000
visual_df = visual_df.replace([np.inf, -np.inf], np.nan)

print(visual_df.shape)
visual_df.head()

## 3. Variables to visualize

In [ ]:
target = 'bike_share_pct'

features = [
    'log_population',
    'n_neighbourhoods',
    'bev_dich_wavg',
    'g_wozbag_wavg',
    'p_koopw_wavg',
    'p_huurw_wavg',
    'g_ink_pi_wavg',
    'g_afs_hp_wavg',
    'g_afs_gs_wavg',
    'g_afs_kv_wavg',
    'g_afs_sc_wavg',
    'g_3km_sc_wavg',
    'ste_mvs_wavg',
    'dwellings_per_1000_residents',
    'pc6_rows_per_1000_residents',
]

plot_df = visual_df[['gm_code', 'Gemeentenaam2025', target] + features].copy()
usable_features = [col for col in features if plot_df[col].notna().any()]
empty_features = sorted(set(features) - set(usable_features))

print('Rows:', len(plot_df))
print('Rows with target:', plot_df[target].notna().sum())
print('Empty variables not plotted in correlation/scatter charts:', empty_features)
plot_df.head()

## 4. Missing values

This figure shows which variables have missing data. For future regression, variables with many missing values may need to be removed or carefully imputed.

In [ ]:
missing = plot_df[[target] + features].isna().mean().mul(100).sort_values(ascending=False)
display(missing.to_frame('missing_pct'))

fig, ax = plt.subplots(figsize=(10, 6))
missing[missing.gt(0)].sort_values().plot(kind='barh', ax=ax, color='#4C78A8')
ax.set_xlabel('Missing values (%)')
ax.set_title('Missingness by Variable')
plt.tight_layout()
plt.savefig(FIGURES / '01_missingness.png', dpi=150)
plt.show()

## 5. Cycling share distribution

This is the potential target variable for later regression. The plot checks its spread and possible outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(plot_df[target], kde=True, ax=axes[0], color='#4C78A8')
axes[0].set_title('Cycling Share Distribution')
axes[0].set_xlabel('Cycling share (%)')

sns.boxplot(x=plot_df[target], ax=axes[1], color='#72B7B2')
axes[1].set_title('Cycling Share Outlier Check')
axes[1].set_xlabel('Cycling share (%)')

plt.tight_layout()
plt.savefig(FIGURES / '02_target_distribution.png', dpi=150)
plt.show()

plot_df[target].describe()

## 6. Predictor distributions

These histograms show whether predictor variables are skewed, concentrated, or affected by outliers.

In [ ]:
feature_subset = [
    'log_population', 'bev_dich_wavg', 'g_wozbag_wavg', 'p_koopw_wavg',
    'p_huurw_wavg', 'g_afs_hp_wavg', 'g_3km_sc_wavg', 'ste_mvs_wavg'
]
feature_subset = [col for col in feature_subset if col in usable_features]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, feature_subset):
    sns.histplot(plot_df[col], kde=True, ax=ax, color='#F58518')
    ax.set_title(col)
    ax.set_xlabel('')
for ax in axes.flat[len(feature_subset):]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / '03_feature_distributions.png', dpi=150)
plt.show()

## 7. Correlation with cycling share

This chart shows simple Pearson correlations with cycling share. It is useful for screening variables before modelling, but it does not prove causality.

In [ ]:
corr_with_target = plot_df[[target] + usable_features].corr(numeric_only=True)[target].drop(target).sort_values()
display(corr_with_target.to_frame('correlation_with_bike_share'))

fig, ax = plt.subplots(figsize=(9, 7))
colors = np.where(corr_with_target.values >= 0, '#54A24B', '#E45756')
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Pearson correlation with bike_share_pct')
ax.set_title('Feature Correlation With Cycling Share')
plt.tight_layout()
plt.savefig(FIGURES / '04_target_correlations.png', dpi=150)
plt.show()

## 8. Predictor correlation heatmap

This visual checks whether predictors are strongly correlated with each other. Strong predictor-to-predictor correlations can make later analysis harder to interpret.

In [ ]:
corr = plot_df[usable_features].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Predictor Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIGURES / '05_predictor_correlation_heatmap.png', dpi=150)
plt.show()

## 9. Scatter plots with trend lines

These plots show the variables with the strongest absolute correlation to cycling share. They help check whether the relationship looks approximately linear.

In [ ]:
top_features = corr_with_target.abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, top_features):
    sns.regplot(data=plot_df, x=col, y=target, ax=ax, scatter_kws={'alpha': 0.65, 's': 35}, line_kws={'color': '#E45756'})
    ax.set_title(f'{target} vs {col}')
for ax in axes.flat[len(top_features):]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / '06_target_vs_key_predictors.png', dpi=150)
plt.show()

top_features